# Day 2: LLM Preprocessing (Vietnamese)

Viet lai descriptions thanh format chuan bang LLM (Groq Batch API).

**Pipeline:** Load from HF Hub -> Test single item -> Batch process 120K items -> Build prompts -> Push to HF Hub

**Model:** `groq/openai/gpt-oss-20b` | **Budget:** ~$11-12 | **Dataset:** `SeanSunny/items_raw_tv_v4` -> `SeanSunny/items_tv_v4`

In [3]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer_vi.batch import Batch
from pricer_vi.items import Item

load_dotenv(override=True)

True

## 1. Load dataset from HuggingFace Hub

In [4]:
dataset = "SeanSunny/items_raw_tv_v4"

train, val, test = Item.from_hub(dataset)
items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 120,000 items
title='Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC TEBAT870' category='Điện Tử - Công Nghệ' price=2376000 full='Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC\nKính chào Quý khách, chào mừng quý khách đến với gian hàng của chúng tôi, chúc Quý khách một ngày tốt lành và mua sắm vui vẻ. Dưới đây là một số thông tin tham khảo về sản phẩm. ĐẶC ĐIỂM: TƯƠNG THÍCH... | Kính chào Quý khách, chào mừng quý khách đến với gian hàng của chúng tôi, chúc Quý khách một ngày tốt lành và mua sắm vui vẻ. Dưới đây là một số thông tin tham khảo về sản phẩm. ĐẶC ĐIỂM: TƯƠNG THÍCH VỚI TẤT CẢ MÃ MÁY CÓ TRONG TÊN SẢN PHẨM THÔNG SỐ KỸ THUẬT Dùng cho tất cả các mã máy có trong tên sản phẩm Công suất: Tiêu chuẩn pin theo máy chênh lệch +/- 5% Điện Áp: Tiêu chuẩn Số Cell: Tiêu chuẩn Loại Pin: Li-on. Thời gian sử dụng cho một lần sạc đầy: 2h – 4h – 6h tùy số Cell và đời máy Hàng mới full box 100%, hoàn toàn tương thích với

In [16]:
print(items[1])

title='Áo len hoodie chất đẹp dày ấm thời trang trẻ trung cho nữ' category='Thời Trang' price=339000 full='Áo len hoodie chất đẹp dày ấm thời trang trẻ trung cho nữ\nMàu sắc tươi sáng, phối màu xinh xắn, trẻ trung.Tỉ mỉ trong từng đường may mũi chỉ.Kiểu dáng thời trang, trẻ trung , dễ phối đồ mặc hàng ngà... | Áo len hoodie chất đẹp dày ấm thời trang trẻ trung cho nữ Màu sắc tươi sáng, phối màu xinh xắn, trẻ trung. Tỉ mỉ trong từng đường may mũi chỉ. Kiểu dáng thời trang, trẻ trung , dễ phối đồ mặc hàng ngày. Chất len dày, ấm, mịn. Kích cỡ: Freesize 60kg Giá sản phẩm trên Tiki đã bao gồm thuế theo luật hiện hành. Bên cạnh đó, tuỳ vào loại sản phẩm, hình thức và địa chỉ giao hàng mà có thể phát sinh thêm chi phí khác như phí vận chuyển, phụ phí hàng cồng kềnh, thuế nhập khẩu (đối với đơn hàng giao từ nước ngoài có giá trị trên 1 triệu đồng)..... | Thương hiệu: LiLiLa | Xuất xứ (Made in): Việt Nam | Sản phẩm có được bảo hành không?: Không' brand='LiLiLa' summary=None prompt=None id=None


In [10]:
print(items[1].title)

Áo len hoodie chất đẹp dày ấm thời trang trẻ trung cho nữ


In [11]:
print(items[1].category)

Thời Trang


In [14]:
print(items[1].price)

339000


In [15]:
print(items[1].full)

Áo len hoodie chất đẹp dày ấm thời trang trẻ trung cho nữ
Màu sắc tươi sáng, phối màu xinh xắn, trẻ trung.Tỉ mỉ trong từng đường may mũi chỉ.Kiểu dáng thời trang, trẻ trung , dễ phối đồ mặc hàng ngà... | Áo len hoodie chất đẹp dày ấm thời trang trẻ trung cho nữ Màu sắc tươi sáng, phối màu xinh xắn, trẻ trung. Tỉ mỉ trong từng đường may mũi chỉ. Kiểu dáng thời trang, trẻ trung , dễ phối đồ mặc hàng ngày. Chất len dày, ấm, mịn. Kích cỡ: Freesize 60kg Giá sản phẩm trên Tiki đã bao gồm thuế theo luật hiện hành. Bên cạnh đó, tuỳ vào loại sản phẩm, hình thức và địa chỉ giao hàng mà có thể phát sinh thêm chi phí khác như phí vận chuyển, phụ phí hàng cồng kềnh, thuế nhập khẩu (đối với đơn hàng giao từ nước ngoài có giá trị trên 1 triệu đồng)..... | Thương hiệu: LiLiLa | Xuất xứ (Made in): Việt Nam | Sản phẩm có được bảo hành không?: Không


In [17]:
print(items[1].brand)

LiLiLa


In [18]:
print(items[1].id)

None


In [19]:
# Assign IDs (required for batch processing custom_id)
for index, item in enumerate(items):
    item.id = index

In [25]:
# Inspect raw data
print(f"Title: {items[1].title}")
print(f"Category: {items[1].category}")
print(f"Price: {items[1].price:,} VND")
print(f"Brand: {items[1].brand}")
print(f"\nFull text ({len(items[1].full)} chars):")
print(items[1].full[:500])

Title: Áo len hoodie chất đẹp dày ấm thời trang trẻ trung cho nữ
Category: Thời Trang
Price: 339,000 VND
Brand: LiLiLa

Full text (842 chars):
Áo len hoodie chất đẹp dày ấm thời trang trẻ trung cho nữ
Màu sắc tươi sáng, phối màu xinh xắn, trẻ trung.Tỉ mỉ trong từng đường may mũi chỉ.Kiểu dáng thời trang, trẻ trung , dễ phối đồ mặc hàng ngà... | Áo len hoodie chất đẹp dày ấm thời trang trẻ trung cho nữ Màu sắc tươi sáng, phối màu xinh xắn, trẻ trung. Tỉ mỉ trong từng đường may mũi chỉ. Kiểu dáng thời trang, trẻ trung , dễ phối đồ mặc hàng ngày. Chất len dày, ấm, mịn. Kích cỡ: Freesize 60kg Giá sản phẩm trên Tiki đã bao gồm thuế theo luậ


In [29]:
# Inspect raw data
print(f"Title: {items[4].title}")
print(f"Category: {items[4].category}")
print(f"Price: {items[4].price:,} VND")
print(f"Brand: {items[4].brand}")
print(f"\nFull text ({len(items[4].full)} chars):")
print(items[4].full[:500])

Title: Dép Đi Trong Nhà Nam Nữ Massage Chân Cực Êm Chân
Category: Thời Trang
Price: 99,750 VND
Brand: giày nữ

Full text (182 chars):
Dép Đi Trong Nhà Nam Nữ Massage Chân Cực Êm Chân
Tăng Kèm Tất TẶNG KÈM 01 ĐÔI TẤT 25kChất liệu Cao su mềm, dẻo đúc nguyên khối nên rất bền chắc nhé Đặc biệt chúng tôi xin được nhắ...


## 2. Test single item with LLM

Test SYSTEM_PROMPT tren 1 item truoc khi batch.

In [33]:
SYSTEM_PROMPT = """Tạo mô tả rõ ràng, ngắn gọn cho một sản phẩm. Chỉ trả lời đúng 2 dòng theo định dạng sau. Không bao gồm mã sản phẩm.
  Mô tả: 1 câu mô tả sản phẩm
  Thông số: 1 câu về tính năng nổi bật"""

MODEL = "groq/openai/gpt-oss-20b"

def build_summary(item, llm_response):
    """Ghep title/category/brand (goc) + mo ta/thong so (LLM) thanh summary."""
    brand = item.brand if item.brand else "Không rõ"
    return (
        f"Tiêu đề: {item.title}\n"
        f"Danh mục: {item.category}\n"
        f"Thương hiệu: {brand}\n"
        f"{llm_response}"
    )

# Test tren 1 item
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[4].full}]
response = completion(messages=messages, model=MODEL, reasoning_effort="low")
llm_text = response.choices[0].message.content

print("=== LLM response ===")
print(llm_text)
print()
print("=== Final summary ===")
summary = build_summary(items[4], llm_text)
print(summary)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

=== LLM response ===
Mô tả: Đệm massage chân cao su mềm, dẻo, bền, phù hợp cho nam nữ, mang lại cảm giác thoải mái và giảm đau.  
Thông số: Chất liệu cao su nguyên khối, khối lượng 25kg, kèm 01 đôi tặng.

=== Final summary ===
Tiêu đề: Dép Đi Trong Nhà Nam Nữ Massage Chân Cực Êm Chân
Danh mục: Thời Trang
Thương hiệu: giày nữ
Mô tả: Đệm massage chân cao su mềm, dẻo, bền, phù hợp cho nam nữ, mang lại cảm giác thoải mái và giảm đau.  
Thông số: Chất liệu cao su nguyên khối, khối lượng 25kg, kèm 01 đôi tặng.

Input tokens: 212
Output tokens: 99
Cost: 0.005 cents


In [27]:
SYSTEM_PROMPT = """Tạo mô tả ngắn gọn cho một sản phẩm. Chỉ trả lời theo định dạng sau. Không bao gồm mã sản phẩm.
Tiêu đề: Viết lại tiêu đề ngắn gọn chính xác
Danh mục: VD Điện tử
Thương hiệu: Tên thương hiệu
Mô tả: 1 câu mô tả sản phẩm
Thông số: 1 câu về tính năng nổi bật"""

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[1].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

Tiêu đề: Áo len hoodie dày ấm cho nữ  
Danh mục: Quần áo nữ  
Thương hiệu: LiLiLa  
Mô tả: Áo len hoodie chất dày, ấm áp với kiểu dáng trẻ trung, màu sắc tươi sáng, dễ phối đồ.  
Thông số: Chất liệu len dày, mịn, kích cỡ Freesize, không bảo hành.

Input tokens: 426
Output tokens: 104
Cost: 0.006 cents


In [28]:
SYSTEM_PROMPT = """Tạo mô tả ngắn gọn cho một sản phẩm. Chỉ trả lời theo định dạng sau. Không bao gồm mã sản phẩm.
Tiêu đề: Viết lại tiêu đề ngắn gọn chính xác
Danh mục: VD Điện tử
Thương hiệu: Tên thương hiệu
Mô tả: 1 câu mô tả sản phẩm
Thông số: 1 câu về tính năng nổi bật"""

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[3].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

Tiêu đề: Combo 2 Sữa Rửa Mặt innisfree Kiểm Soát Nhờn Tro Núi Lửa & BHA  
Danh mục: Sản phẩm chăm sóc da  
Thương hiệu: innisfree  
Mô tả: Sữa rửa mặt giúp kiểm soát dầu, làm sạch sâu và se khít lỗ chân lông.  
Thông số: Chứa BHA và đá tro núi lửa, phù hợp với da dầu/mụn, dung tích 2x150g.

Input tokens: 916
Output tokens: 127
Cost: 0.011 cents


In [30]:
SYSTEM_PROMPT = """Tạo mô tả ngắn gọn cho một sản phẩm. Chỉ trả lời theo định dạng sau. Không bao gồm mã sản phẩm.
Tiêu đề: Viết lại tiêu đề ngắn gọn chính xác
Danh mục: VD Điện tử
Thương hiệu: Tên thương hiệu
Mô tả: 1 câu mô tả sản phẩm
Thông số: 1 câu về tính năng nổi bật"""

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[4].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

Tiêu đề: Đệm Đi Trong Nhà Massage Chân Cực Êm Chân  
Danh mục: Đệm và Thảm Massage  
Thương hiệu: Chân  
Mô tả: Đệm massage chân nhẹ nhàng giúp giảm đau, tăng tuần hoàn, thích hợp cho cả nam và nữ sử dụng tại nhà.  
Thông số: Chất liệu cao su mềm, dẻo đúc nguyên khối, bền chắc và khuyến mãi kèm 01 đôi tặng 25k.

Input tokens: 234
Output tokens: 173
Cost: 0.007 cents


## make_jsonl + make_file (tao 0_10.jsonl de test batch):

In [39]:
import json
import os
from groq import Groq

groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
BATCH_MODEL = "openai/gpt-oss-20b"

os.makedirs("jsonl_vi", exist_ok=True)

def make_jsonl(item):
    body = {
        "model": BATCH_MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item.full},
        ],
        "reasoning_effort": "low",
    }
    line = {
        "custom_id": str(item.id),
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": body,
    }
    return json.dumps(line, ensure_ascii=False)

def make_file(start, end, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

make_file(11, 20, "jsonl_vi/11_20.jsonl")
print("Created jsonl_vi/11_20.jsonl")


Created jsonl_vi/11_20.jsonl


In [40]:
# Upload
with open("jsonl_vi/11_20.jsonl", "rb") as f:
    file_response = groq_client.files.create(file=f, purpose="batch")
file_id = file_response.id
print(f"Uploaded: {file_id}")

# Submit batch
batch_response = groq_client.batches.create(
    completion_window="24h",
    endpoint="/v1/chat/completions",
    input_file_id=file_id,
)
print(f"Batch: {batch_response.id}, status: {batch_response.status}")


Uploaded: file_01knztd7e6fk9r6ap17hs78m22
Batch: batch_01knztd7q8f5ebzwxzt6eap3fz, status: validating


In [42]:
# Check status (chay lai cho den khi completed)
result = groq_client.batches.retrieve(batch_response.id)
print(f"Status: {result.status}")
if result.status == "completed":
    print(f"Output file: {result.output_file_id}")

Status: completed
Output file: file_01knztdckveags3pmng0dbdjnk


In [43]:
# Fetch results + build summaries
output = groq_client.files.content(result.output_file_id)
output.write_to_file("jsonl_vi/batch_results.jsonl")

with open("jsonl_vi/batch_results.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        llm_text = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = build_summary(items[id], llm_text)

# Xem ket qua
for i in range(10):
    print(f"--- Item {i} ({items[i].category}, {items[i].price:,} VND) ---")
    print(items[i].summary)
    print()

--- Item 0 (Điện Tử - Công Nghệ, 2,376,000 VND) ---
Tiêu đề: Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC TEBAT870
Danh mục: Điện Tử - Công Nghệ
Thương hiệu: TEEMO PC
Mô tả: Pin Li‑on mới, tương thích hoàn toàn với Dell Vostro 14 5459, có thể cung cấp thời gian sử dụng lên đến 6 giờ tùy cấu hình.  
Thông số: Công suất ±5%, 2‑6 giờ sạc đầy, bảo hành 6‑12 tháng, bao bì nguyên mẫu 100% và hỗ trợ đổi mới nếu lỗi trong thời gian bảo hành.

--- Item 1 (Thời Trang, 339,000 VND) ---
Tiêu đề: Áo len hoodie chất đẹp dày ấm thời trang trẻ trung cho nữ
Danh mục: Thời Trang
Thương hiệu: LiLiLa
Mô tả: Áo len hoodie nữ màu sắc tươi sáng, thiết kế trẻ trung, chất liệu len dày ấm, dễ phối đồ hàng ngày.  
Thông số: Kiểu dáng thời trang, tỉ mỉ đường may mũi chỉ, có kích cỡ Freesize, phù hợp cho trọng lượng lên tới 60kg.

--- Item 2 (Bách Hóa, 78,000 VND) ---
Tiêu đề: Trà lá xanh hương lá dứa Trần Quang (gói 500gr)
Danh mục: Bách Hóa
Thương hiệu: Trần Quang
Mô tả: Trà 

## 3. Batch processing (Groq Batch API)

Chia 120K items thanh batches 1,000 items/batch. Submit len Groq, doi ket qua.

**Chi phi uoc tinh:** ~$11-12 cho 120K items.

In [ ]:
Batch.create(items)

In [ ]:
Batch.run()

In [ ]:
# Chay cell nay nhieu lan cho den khi tat ca batches hoan thanh
Batch.fetch()

In [ ]:
# Save state (de resume neu can)
Batch.save()

In [ ]:
# Load state (neu can resume tu session truoc)
# Batch.load(items)

## 4. Kiem tra ket qua

In [ ]:
# Kiem tra co item nao thieu summary khong
missing = [i for i, item in enumerate(items) if not item.summary]
print(f"Missing summaries: {len(missing)}")
if missing:
    print(f"First 10 missing IDs: {missing[:10]}")

In [ ]:
# Xem vi du summary
for i in [0, 100, 1000, 5000, 50000]:
    if i < len(items) and items[i].summary:
        print(f"--- Item {i} ({items[i].category}, {items[i].price:,} VND) ---")
        print(items[i].summary)
        print()

## 5. Build prompts va clean up

Tao prompt tu summary cho fine-tuning. Sau do xoa cac field khong can (full, brand, id).

In [ ]:
# Build prompt from summary for each item
for item in items:
    if item.summary:
        item.make_prompt(item.summary)

# Verify
print(items[0].prompt[:300] if items[0].prompt else "No prompt")

In [ ]:
# Clean up: remove fields not needed in final dataset
for item in items:
    item.full = None
    item.brand = None
    item.id = None

In [ ]:
# Verify final schema
print(items[0].model_dump())

## 6. Push to HuggingFace Hub

Dataset final: `SeanSunny/items_tv_v4`

Schema: title, category, price, summary, prompt (full/brand/id = None)

In [ ]:
username = "SeanSunny"
output_dataset = f"{username}/items_tv_v4"

# Split back to train/val/test (same sizes as Day 1: 110K/5K/5K)
train = items[:110_000]
val = items[110_000:115_000]
test = items[115_000:]

print(f"Train: {len(train):,}, Val: {len(val):,}, Test: {len(test):,}")
Item.push_to_hub(output_dataset, train, val, test)
print(f"Pushed to {output_dataset}")

## Done!

Dataset `SeanSunny/items_tv_v4` da co summary va prompt. San sang cho Day 3 (Baseline ML) va Day 4 (DNN + Frontier LLM).